# Gold: Sales by Country

**Objetivo:** consolidar métricas de vendas por país e por dia, em inglês (valores e nomes de colunas), para consumo executivo (relatório CFO). Fonte de verdade para análises por país.

**Fonte:** tabela Delta `silver.fato_vendas`, enriquecida com `silver.produtos` (nome em inglês) e tradução de país.

**Destino:** tabela Delta `gold.sales_by_country`.

**Granularidade:** país + dia (`period`).

**Observação:** dias anteriores à existência de `dim_cambio` (sem cotação disponível) resultam em `total_usd` nulo para aquele país/dia — reflexo direto da limitação já documentada na Silver, não um erro deste notebook.

In [0]:
# Imports e configuração do schema

from pyspark.sql.functions import col, sum as spark_sum, when

spark.sql("CREATE SCHEMA IF NOT EXISTS poc_latam_food.gold")

print("Schema 'gold' verificado/criado com sucesso.")

In [0]:
# Leitura da Silver + tradução do país

df_fato_vendas = spark.table("poc_latam_food.silver.fato_vendas")

df_vendas_traduzido = (
    df_fato_vendas
    .withColumn(
        "country",
        when(col("pais") == "BRASIL", "Brazil")
        .when(col("pais") == "ARGENTINA", "Argentina")
        .when(col("pais") == "MEXICO", "Mexico")
        .otherwise(col("pais"))
    )
)

df_vendas_traduzido.select("pais", "country").distinct().display()

In [0]:
# Agregação por país e dia (com arredondamento correto)

from pyspark.sql.functions import round as spark_round

df_gold_by_country = (
    df_vendas_traduzido
    .groupBy("country", "moeda", col("data_venda").alias("period"))
    .agg(
        spark_round(spark_sum("valor_total_moeda_local"), 2).cast("decimal(15,2)").alias("total_local_currency"),
        spark_round(spark_sum("valor_total_usd"), 2).cast("decimal(15,2)").alias("total_usd")
    )
    .withColumnRenamed("moeda", "local_currency_code")
    .select("country", "period", "total_local_currency", "local_currency_code", "total_usd")
    .orderBy("period", "country")
)

df_gold_by_country.display()



In [0]:
# Gravação - overwrite completo (recálculo), permitindo mudança de schema

(
    df_gold_by_country
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("poc_latam_food.gold.sales_by_country")
)

print("Gold sales_by_country concluído.")

In [0]:
# Validação visual - gold.sales_by_country

spark.table("poc_latam_food.gold.sales_by_country").display()

In [0]:
# Visão complementar - somatório total por país (consulta, apenas moeda local)

df_gold_by_country.groupBy("country", "local_currency_code").agg(
    spark_round(spark_sum("total_local_currency"), 2).alias("total_local_currency_geral")
).orderBy("country").display()